1. evaluate a bunch of CNNs with a couple of different metanetworks
2. get intra-CNN variance to analyze how metanetwork-invariant the results are
3. if this is the case, see if we can cluster bad vs. good CNNs for unlearning
4. Analyze why this is

# 0. Train a couple of metanetworks ✅

In [ ]:
from cnn_surgery.utils.load_dataset import load_multi_stage_dataset
from cnn_surgery.lenses.regressor_lens import get_regressor_lens
import torch
from tqdm import tqdm
import os
import json
import pickle

datasets = ["mnist", "fashion_mnist", "cifar10"]

for dataset in datasets:
    train, val, _ = load_multi_stage_dataset(include_test=False, dataset=dataset).values() # type: ignore

    weights_train = train[0]
    weights_val = val[0]

    accuracies_train = train[1]
    accuracies_val = val[1]

    configs_train = train[2]
    configs_val = val[2]

    for i in range(5):
        MetaNetwork, metrics = get_regressor_lens(
            weights_train,
            accuracies_train,
            weights_val,
            accuracies_val,
            device="cpu",
            return_metrics=True,
            verbose=False,
        )  # type: ignore

        # make sure parent directory exists
        os.makedirs("../models/good_bad_experiment_2", exist_ok=True)
        torch.save(MetaNetwork.state_dict(), f"../models/good_bad_experiment_2/{dataset}_metanetwork_{i}.pt")
        # should've saved them as pickles
        # with open(f"../models/good_bad_experiment_2/{dataset}_metanetwork_{i}.pkl", "wb") as f:
        #     pickle.dump(MetaNetwork, f)

        # save metrics
        metrics_dict = {
            "mse_train": metrics[0][0],
            "mae_train": metrics[0][1],
            "mse_val": metrics[1][0],
            "mae_val": metrics[1][1],
            "r2_val": metrics[2],
        }

        with open(f"../models/good_bad_experiment_2/{dataset}_metanetwork_{i}_metrics.json", "w") as f:
            json.dump(metrics_dict, f, indent=2)

## 0.1 Analyze the metanetworks (are they all good?) ✅

plot mean and std of the metanetworks per dataset

In [ ]:
import json
import pandas as pd
import plotly.express as px

metrics_df = pd.DataFrame()

datasets = ["mnist", "fashion_mnist", "cifar10"]
for dataset in datasets:
    for i in range(5):
        metrics_path = f"../models/good_bad_experiment_2/{dataset}_metanetwork_{i}_metrics.json"
        with open(metrics_path, "r") as f:
            metrics = json.load(f)
            
            # add the json data to the dataframe
            row = {"dataset": dataset, "metanetwork_idx": i} 
            row.update(metrics)
            metrics_df = pd.concat([metrics_df, pd.DataFrame([row])], ignore_index=True)


display(metrics_df)
metrics_df.describe()

# plot r2_val per dataset (narrower figure)
fig = px.box(metrics_df, x="dataset", y="r2_val", points="all", title="Metanetwork R2 on validation set per dataset")
fig.update_layout(width=500, height=400)
fig.show()

# 1. Evaluate a bunch of CNNs with these metanetworks ✅ (200)
# 2. Get intra-CNN variance to analyze how metanetwork-invariant the results are

## 2.1 create a master dataframe ✅

In [ ]:
import pandas as pd
import numpy as np
import ast
from cnn_surgery.utils import metrics

datasets = ["mnist", "fashion_mnist", "cifar10"]

master_df = pd.DataFrame()

for dataset in datasets:
    for i in range(5):
        for j in range(10):
            path = f"../experiments/good-bad/{dataset}_eval_results_meta_{i}_class_{j}.csv"
            df = pd.read_csv(path)

            # calculate metrics
            df['max_difference'] = df.apply(lambda row: metrics.max_difference(ast.literal_eval(row['original_accuracy']), ast.literal_eval(row['accuracy_after']), row['target_class']), axis=1)
            df['clipped_negative_mean_difference'] = df.apply(lambda row: metrics.clipped_negative_mean_difference(ast.literal_eval(row['original_accuracy']), ast.literal_eval(row['accuracy_after']), row['target_class']), axis=1)
            #df['relative_clipped_negative_mean_difference'] = df.apply(lambda row: metrics.clipped_negative_mean_difference(ast.literal_eval(row['original_accuracy']), ast.literal_eval(row['accuracy_after']), row['target_class'], proportional=True), axis=1)
            df['target_difference'] = df.apply(lambda row: metrics.target_difference(ast.literal_eval(row['original_accuracy']), ast.literal_eval(row['accuracy_after']), row['target_class']), axis=1)
            df['mse_init_pred'] = df.apply(lambda row: ((np.array(ast.literal_eval(row['init_pred'])) - np.array(ast.literal_eval(row['original_accuracy'])))**2).mean(), axis=1)
            df['metanetwork_idx'] = i
            df['metanetwork_path'] = f"../models/good_bad_experiment_2/{dataset}_metanetwork_{i}.pt"
            df['metanetwork_metrics_path'] = f"../models/good_bad_experiment_2/{dataset}_metanetwork_{i}_metrics.json"
            
            master_df = pd.concat([master_df, df], ignore_index=True)

display(master_df)
master_df.describe()


In [ ]:
import plotly.express as px

# drop all rows with dataset mnist or cifar10
master_df = master_df[master_df['dataset'] != 'mnist']
master_df = master_df[master_df['dataset'] != 'cifar10']

# only use rows with top 10% average initial accuracy
master_df['avg_initial_accuracy'] = master_df['original_accuracy'].apply(lambda x: np.mean(ast.literal_eval(x)))
master_df = master_df[master_df['avg_initial_accuracy'] >= master_df['avg_initial_accuracy'].quantile(0.9)]

# Group by model_idx, dataset, target_class and calculate mean and std over metanetwork_idx
grouped_stats = master_df.groupby(['model_idx', 'dataset', 'target_class']).agg({
    'max_difference': ['mean', 'std'],
    'clipped_negative_mean_difference': ['mean', 'std'],
    'target_difference': ['mean', 'std']
}).reset_index()

# Flatten column names
grouped_stats.columns = ['_'.join(col).strip('_') for col in grouped_stats.columns]
display(grouped_stats)

# Melt the dataframe to get std columns in long format for plotting
std_cols = ['max_difference_std', 'clipped_negative_mean_difference_std', 'target_difference_std']
melted_std = grouped_stats.melt(
    id_vars=['model_idx', 'dataset', 'target_class'],
    value_vars=std_cols,
    var_name='metric',
    value_name='std_value'
)

## STDs
# Clean up metric names for display
melted_std['metric'] = melted_std['metric'].str.replace('_std', '')

# Create boxplot
fig = px.scatter(melted_std, x='metric', y='std_value', color='dataset', 
             title='Standard Deviation of Metrics Across Metanetworks',
             labels={'std_value': 'Standard Deviation', 'metric': 'Metric'})
fig.show()

## means
# Melt the dataframe to get mean columns in long format for plotting
mean_cols = ['max_difference_mean', 'clipped_negative_mean_difference_mean', 'target_difference_mean']
melted_mean = grouped_stats.melt(
    id_vars=['model_idx', 'dataset', 'target_class'],
    value_vars=mean_cols,
    var_name='metric',
    value_name='mean_value'
)

# Clean up metric names for display
melted_mean['metric'] = melted_mean['metric'].str.replace('_mean', '')

# Create boxplot
fig = px.scatter(melted_mean, x='metric', y='mean_value', color='dataset', 
             title='Mean of Metrics Across Metanetworks',
             labels={'mean_value': 'Mean', 'metric': 'Metric'})
fig.show()

## Detailed boxplots per metanetwork
# Plot max_difference per metanetwork (5 metanetworks x 3 datasets = 15 boxplots)
# fig = px.scatter(master_df[master_df['model_idx'] == 3][master_df['target_class'] == 1], x='metanetwork_idx', y='max_difference', color='model_idx',
#              title='Max Difference per Metanetwork',
#              labels={'max_difference': 'Max Difference', 'metanetwork_idx': 'Metanetwork Index'},
#              hover_data=['target_class'])
# fig.update_yaxes(range=[-1, 1])
# fig.show()

# # Plot clipped_negative_mean_difference per metanetwork
# fig = px.scatter(master_df[master_df['target_class'] == 1], x='metanetwork_idx', y='clipped_negative_mean_difference', color='dataset',
#              title='Clipped Negative Mean Difference per Metanetwork',
#              labels={'clipped_negative_mean_difference': 'Clipped Negative Mean Difference', 'metanetwork_idx': 'Metanetwork Index'},
#              hover_data=['target_class'],
#              animation_frame='model_idx')
# fig.update_yaxes(range=[-1, 1])
# fig.layout.updatemenus[0].buttons[0].args[1]['frame']['duration'] = 100
# fig.layout.updatemenus[0].buttons[0].args[1]['transition']['duration'] = 0
# fig.show()

# # Plot target_difference per metanetwork
# fig = px.scatter(master_df[master_df['model_idx'] == 3][master_df['target_class'] == 1], x='metanetwork_idx', y='target_difference', color='model_idx',
#              title='Target Difference per Metanetwork',
#              labels={'target_difference': 'Target Difference', 'metanetwork_idx': 'Metanetwork Index'},
#              hover_data=['target_class'])
# fig.update_yaxes(range=[-1, 1])
# fig.show()

# ## Quantify the difference in performance (expressed in the three metrics) of the five metanetworks
# # Calculate mean performance metrics per metanetwork and dataset
# metanetwork_performance = master_df.groupby(['metanetwork_idx', 'dataset']).agg({
#     'max_difference': 'std',
#     'clipped_negative_mean_difference': 'std',
#     'target_difference': 'std'
# }).reset_index()

# display(metanetwork_performance)

# # Calculate summary statistics across metanetworks per dataset
# print("\nPerformance variation across metanetworks per dataset:")
# performance_summary = metanetwork_performance.groupby('dataset').agg({
#     'max_difference': ['mean', 'std', 'min', 'max'],
#     'clipped_negative_mean_difference': ['mean', 'std', 'min', 'max'],
#     'target_difference': ['mean', 'std', 'min', 'max']
# })
# display(performance_summary)

# # ANOVA-style analysis: variance between metanetworks vs within
# print("\nCoefficient of Variation (std/|mean|) across metanetworks per dataset:")
# cv_stats = metanetwork_performance.groupby('dataset').apply(
#     lambda x: pd.Series({
#         'max_difference_cv': x['max_difference'].std() / abs(x['max_difference'].mean()) if x['max_difference'].mean() != 0 else 0,
#         'clipped_negative_mean_difference_cv': x['clipped_negative_mean_difference'].std() / abs(x['clipped_negative_mean_difference'].mean()) if x['clipped_negative_mean_difference'].mean() != 0 else 0,
#         'target_difference_cv': x['target_difference'].std() / abs(x['target_difference'].mean()) if x['target_difference'].mean() != 0 else 0
#     })
# )
# display(cv_stats)

What are characteristics of the outliers?
gemiddelde performance per metanetwerk